# Autonomous-vehicle GPS spoofing detection

**Kościuszkon 2026 — Honeywell Theme #2**

The same physics that protect a drone protect a self-driving car: a sudden
position jump, an impossible velocity, or a GPS-quality cliff are the
signatures of a spoofing attack. The University of Arizona ACL-Rover dataset
records GPS telemetry while the testbed is being attacked by a software-defined
radio.

This notebook replays the analysis from `01_UAV_Analysis.ipynb` on the AV
dataset (~62k samples). At the end we cross-compare the two domains to argue
that the same hybrid approach generalises across vehicle types.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

sys.path.append(str(pathlib.Path.cwd().parent))
from gps_detection_utils import train_rf, evaluate, plot_results, compare_methods

RNG = np.random.default_rng(42)
plt.rcParams['figure.dpi'] = 90

## 1. Data — real CSV or labelled simulation

Same fallback discipline as the UAV notebook: try the AV-GPS-Dataset CSV
first, otherwise simulate a labelled urban drive whose attack windows produce
position drift and GPS-quality degradation.

In [ ]:
def simulate_av_drive(n=15000, attack_rate=0.254, rng=RNG):
    """Simulate an urban drive with discrete spoofing windows."""
    t = np.arange(n)
    base_lat = 33.4484 + 5e-3 * np.sin(2 * np.pi * t / 3000)
    base_lon = -112.074 + 5e-3 * np.sin(2 * np.pi * t / 4000)
    speed   = 8 + 4 * np.sin(2 * np.pi * t / 600) + rng.normal(0, 0.5, n)  # ~30 km/h cruise
    speed   = np.clip(speed, 0, 25)

    is_attack = np.zeros(n, dtype=bool)
    target = int(n * attack_rate)
    while is_attack.sum() < target:
        s = int(rng.integers(0, n - 600));  L = int(rng.integers(200, 500))
        is_attack[s:s + L] = True

    lat = base_lat + rng.normal(0, 1e-5, n)
    lon = base_lon + rng.normal(0, 1e-5, n)
    course = np.degrees(np.arctan2(np.gradient(base_lon), np.gradient(base_lat))) % 360
    hdop = np.abs(rng.normal(1.5, 0.3, n))
    vdop = np.abs(rng.normal(2.0, 0.4, n))
    sats = rng.integers(8, 14, n).astype(float)
    locks = sats - rng.integers(0, 2, n)

    if is_attack.any():
        k = is_attack.sum()
        lat[is_attack]   += np.cumsum(rng.normal(0, 5e-6, k))
        lon[is_attack]   += np.cumsum(rng.normal(0, 5e-6, k))
        speed[is_attack] += rng.normal(0, 4, k)
        hdop[is_attack]  *= rng.uniform(2, 4, k)
        vdop[is_attack]  *= rng.uniform(2, 4, k)
        sats[is_attack]   = np.clip(sats[is_attack] - rng.integers(2, 5, k), 3, 14)
        locks[is_attack]  = np.clip(locks[is_attack] - rng.integers(2, 5, k), 1, 12)

    return pd.DataFrame({
        'recording': 'sim_drive',
        'GPS Latitude': lat, 'GPS Longitude': lon,
        'Velocity (m/s)': np.clip(speed, 0, 60),
        'GPS Course': course,
        'GPS HDOP': hdop, 'GPS VDOP': vdop,
        'Satellite Count': sats, 'Satellite Locks': locks,
        'Data Type': is_attack.astype(int),
    })


def load_av():
    csv_path = '../../AV-GPS-Dataset/AV-GPS-Dataset-1.csv'
    try:
        df = pd.read_csv(csv_path)
        df['recording'] = 'real_drive'
        print(f"Loaded real AV dataset: {len(df):,} samples")
    except FileNotFoundError:
        df = simulate_av_drive()
        print(f"Real CSV not found — using labelled simulation: {len(df):,} samples")
    return df

df = load_av()
df.head(3)

## 2. Exploratory data analysis

In [ ]:
print(f"Shape:           {df.shape}")
print(f"Missing values:  {int(df.isna().sum().sum())}")
print(f"Attack rate:     {df['Data Type'].mean():.1%}")
print("\nSummary statistics for selected GPS quality features:")
df[['Velocity (m/s)', 'GPS HDOP', 'GPS VDOP', 'Satellite Count', 'Satellite Locks']].describe().round(2)

**What separates attack from normal?** Below: class balance, HDOP shift,
satellite drop-out, and per-step position move (in metres) — the same set as
the UAV notebook, computed on AV columns.

In [ ]:
g = df.groupby('recording', sort=False)
step_m = np.hypot(g['GPS Latitude'].diff(), g['GPS Longitude'].diff()).fillna(0) * 111_320
normal = df['Data Type'] == 0

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ax[0, 0].pie(df['Data Type'].value_counts().sort_index(),
             labels=['benign', 'attack'], autopct='%1.1f%%',
             colors=['#7fbf7f', '#d65f5f'])
ax[0, 0].set_title('Class balance')

ax[0, 1].hist([df.loc[normal, 'GPS HDOP'], df.loc[~normal, 'GPS HDOP']],
              bins=40, label=['benign', 'attack'], color=['#7fbf7f', '#d65f5f'])
ax[0, 1].set_title('HDOP distribution'); ax[0, 1].set_xlabel('HDOP'); ax[0, 1].legend()

ax[1, 0].hist([df.loc[normal, 'Satellite Count'], df.loc[~normal, 'Satellite Count']],
              bins=range(3, 16), label=['benign', 'attack'], color=['#7fbf7f', '#d65f5f'])
ax[1, 0].set_title('Satellite count'); ax[1, 0].legend()

ax[1, 1].hist([step_m[normal].clip(0, 5), step_m[~normal].clip(0, 5)],
              bins=40, label=['benign', 'attack'], color=['#7fbf7f', '#d65f5f'])
ax[1, 1].set_title('Per-step position move (m, clipped at 5)')
ax[1, 1].set_xlabel('metres / sample'); ax[1, 1].legend()

plt.tight_layout(); plt.show()

## 3. Feature engineering

Same idea as the UAV notebook: derivatives are taken **within each
recording** so they are physically meaningful and there is no label leakage
(`recording` is the session identifier, `Data Type` is the per-row label).

In [ ]:
g = df.groupby('recording', sort=False)
df['speed_kmh']    = df['Velocity (m/s)'] * 3.6
df['lat_delta']    = g['GPS Latitude'].diff().abs().fillna(0)
df['lon_delta']    = g['GPS Longitude'].diff().abs().fillna(0)
df['step_metres']  = np.hypot(df['lat_delta'], df['lon_delta']) * 111_320
df['acceleration'] = df['Velocity (m/s)'].diff().abs().fillna(0)
df['heading_delta'] = g['GPS Course'].diff().abs().fillna(0)
df['heading_delta'] = np.where(df['heading_delta'] > 180,
                               360 - df['heading_delta'], df['heading_delta'])
df['gps_quality']  = df['GPS HDOP'] + df['GPS VDOP']

FEATURES = ['Velocity (m/s)', 'speed_kmh', 'acceleration',
            'step_metres', 'heading_delta',
            'GPS HDOP', 'GPS VDOP', 'gps_quality',
            'Satellite Count', 'Satellite Locks']
print(f"{len(FEATURES)} features used")
df[FEATURES].describe().round(2)

## 4. Methods

* **Baseline (physics rules)**: speed > 300 km/h, single-step move > 50 m,
  GPS-quality > 6, or sharp heading change > 90°.
* **RandomForest** (100 trees, depth 12, balanced classes, stratified 70/30
  split) — same configuration as the UAV notebook so the comparison is fair.

In [ ]:
# Physics-rules baseline
df['rule_alarm'] = (
    (df['speed_kmh']     > 300) |
    (df['step_metres']   > 50)  |
    (df['gps_quality']   > 6)   |
    (df['heading_delta'] > 90)
).astype(int)
rule_acc = (df['rule_alarm'] == df['Data Type']).mean()
print(f"Rules-only accuracy:  {rule_acc:.4f}")

# RandomForest on the same features
model, X_te, y_te = train_rf(df, FEATURES, 'Data Type')
ml_metrics, y_pred, _ = evaluate(model, X_te, y_te, label="AV — RandomForest on test set")

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
plot_results(y_te, y_pred, importance, ['benign', 'attack'], title='AV')
print("\nClassification report on test set:")
print(classification_report(y_te, y_pred, target_names=['benign', 'attack']))

## 5. Comparison & cross-domain conclusion

In [ ]:
compare_methods(ml_metrics, rule_acc, len(df), df['Data Type'].mean(), domain="AV")

## Results Summary

In [ ]:
# Final Results using utility function
print_final_results(df_av, av_ml_accuracy, av_rule_accuracy, "AV", "Data Type")